# Taller 04 - Clustering

Este notebook sirve como base para resolver un ejercicio de clustering. Puedes reemplazar el archivo de datos por el de tu taller y ejecutar las celdas en orden.

## Objetivo
- Explorar el conjunto de datos
- Elegir variables relevantes
- Aplicar k-means y comparar con otros algoritmos
- Interpretar los clusters y sus resultados

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
%matplotlib inline

## 1. Cargar el dataset

Ajusta la ruta del archivo si tu dataset está en otra carpeta. Si no encuentras un CSV, el notebook usa un dataset de ejemplo del paquete scikit-learn para que puedas probarlo completo.

In [ ]:
candidate_paths = [
    './data.csv',
    './dataset.csv',
    './datos.csv',
    './train.csv',
    'data.csv',
    'dataset.csv',
    'datos.csv',
    'train.csv'
]

df = None
for path in candidate_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'Dataset cargado desde: {path}')
        break

if df is None:
    from sklearn.datasets import load_wine
    X, y = load_wine(return_X_y=True, as_frame=True)
    df = pd.concat([X, pd.Series(y, name='label')], axis=1)
    print('No se encontró un CSV local. Se usó el dataset de ejemplo Wine de scikit-learn.')

df.head()

## 2. Exploración inicial

Revisa la estructura del dataset, los tipos de variables y si hay valores faltantes.

In [ ]:
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print('\nTipos de columnas:')
print(df.dtypes)
print('\nValores faltantes:')
print(df.isnull().sum())
print('\nResumen estadístico:')
display(df.describe(include='all').T)

## 3. Preparación de datos

Se seleccionan las columnas numéricas para aplicar clustering y se estandarizan para evitar que variables con mayor escala dominen el análisis.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'label' in numeric_cols:
    numeric_cols.remove('label')

X = df[numeric_cols].copy()
print(f'Variables numéricas usadas: {numeric_cols}')
print(f'Forma de X: {X.shape}')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=numeric_cols)
X_scaled.head()

## 4. Método del codo (Elbow) para K-means

El objetivo es identificar un número razonable de clusters usando la suma de cuadrados dentro del cluster (WCSS).

In [ ]:
inertia = []
for k in range(1, 11):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertia.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), inertia, marker='o', linewidth=2)
plt.title('Método del codo para K-means')
plt.xlabel('Número de clusters (k)')
plt.ylabel('WCSS')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Aplicar K-means

Selecciona el valor de k que consideres más adecuado, por ejemplo con la gráfica del codo y/o el coeficiente de silueta.

In [ ]:
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df['cluster_kmeans'] = clusters
print('Distribución de clusters:')
print(df['cluster_kmeans'].value_counts().sort_index())

silhouette = silhouette_score(X_scaled, clusters)
print(f'\nSilhouette Score: {silhouette:.4f}')

df[['cluster_kmeans'] + numeric_cols[:5]].head()

## 6. Comparar con otros algoritmos

- DBSCAN es útil cuando no sabes cuántos clusters existen y hay ruido.
- Agglomerative Clustering sirve para comparar una estrategia jerárquica.

In [ ]:
dbscan = DBSCAN(eps=1.5, min_samples=5)
labels_dbscan = dbscan.fit_predict(X_scaled)
df['cluster_dbscan'] = labels_dbscan
print('Distribución DBSCAN:')
print(df['cluster_dbscan'].value_counts().sort_index())

agg = AgglomerativeClustering(n_clusters=k)
labels_agg = agg.fit_predict(X_scaled)
df['cluster_agg'] = labels_agg
print('\nDistribución clustering jerárquico:')
print(df['cluster_agg'].value_counts().sort_index())

## 7. Visualización en 2D

Se reduce la dimensionalidad a dos componentes para graficar los clusters.

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster_kmeans'], cmap='viridis', s=50, alpha=0.8)
plt.title('Clusters obtenidos con K-means (PCA 2D)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.2)
plt.show()

## 8. Interpretación de resultados

En esta sección debes describir qué caracteriza a cada cluster y decidir cuál algoritmo resulta más apropiado para este problema.

In [ ]:
cluster_summary = df.groupby('cluster_kmeans')[numeric_cols].mean().round(3)
print('Promedio por cluster:')
display(cluster_summary)

print('\nResumen de interpretación:')
print('- Cluster 0: observa los valores promedio para identificar su perfil principal.')
print('- Cluster 1: compara con el resto para describir diferencias.')
print('- Cluster 2: interpreta la separación lógica del problema.')

## 9. Conclusiones

Escribe aquí tus conclusiones finales:
- ¿Cuál es el mejor número de clusters?
- ¿Qué algoritmo funciona mejor en este caso?
- ¿Qué significado tienen los clusters en el problema analizado?